In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1
    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = ['0_2','0_3','0_4','0_5','2_3','2_4','2_5','3_4','3_5','4_5']

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

['0_2', '0_3', '0_4', '0_5', '2_3', '2_4', '2_5', '3_4', '3_5', '4_5']

# ROC

In [4]:
from sklearn.metrics import roc_curve
exp_test = []
y_true_all_exp_test = []
for i in range(len(filenames)):
    exp_test.append(pd.read_csv(f'test_{filenames[i]}_GMM_BIC_1_10_scores.csv'))
    test_encoded_df = pd.read_csv(f'test_encoded_CADE_cod_joined_attacks_pick_class_squared_normalized_2_hidden_{filenames[i]}_hidden.csv')
    y_true_all_exp_test.append(test_encoded_df['Label'].values.tolist())


In [5]:
from sklearn.metrics import classification_report
ths = [21]
f1s = [[]]
cd_test_index = []
for i in range(len(exp_test)):
    hidden_classes = list(map(int, filenames[i].split('_'))) # Classes ocultas do treinamento
    index = 0
    cd_test_index.append([])
    for th in ths:
        y_pred = []
        for j in range(len(exp_test[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_test[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_test[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
                cd_test_index[-1].append(j)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_test[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][hidden_classes].sum()
        fp = cm[-1].sum() - tp
        fn = cm.iloc[hidden_classes].values.sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_test[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0_2


,0,1,2,3,4,5,-1
0,0,35,0,0,0,0,252751
1,0,142359,0,20,2,0,18289
2,0,442,0,2620,0,0,99819
3,0,0,0,76088,0,0,99
4,0,0,0,0,50517,0,6721
5,0,0,0,0,5,23,157
-1,0,0,0,0,0,0,0


th: 21 hidden: 0_2 acc:0.9563610571323508 recall:0.9912924167831145 precision:0.933129717655279 f1:0.9613321281576217
MULTICLASS DETECTION


d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dow

              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         0
           0       0.00      0.00      0.00    252786
           1       1.00      0.89      0.94    160670
           2       0.00      0.00      0.00    102881
           3       0.97      1.00      0.98     76187
           4       1.00      0.88      0.94     57238
           5       1.00      0.12      0.22       185

    accuracy                           0.41    649947
   macro avg       0.57      0.41      0.44    649947
weighted avg       0.45      0.41      0.43    649947

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0_3


...

d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dow


[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 4_5


,0,1,2,3,4,5,-1
0,243622,0,0,0,0,0,9164
1,44,18603,1,0,0,0,142022
2,0,0,90819,0,0,0,12062
3,0,0,1,67107,0,0,9079
4,0,0,0,0,0,0,57238
5,0,0,0,0,0,0,185
-1,0,0,0,0,0,0,0


th: 21 hidden: 4_5 acc:0.7348599193472699 recall:1.0 precision:0.24993688792165397 f1:0.3999192124607815
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         0
           0       1.00      0.96      0.98    252786
           1       1.00      0.12      0.21    160670
           2       1.00      0.88      0.94    102881
           3       1.00      0.88      0.94     76187
           4       0.00      0.00      0.00     57238
           5       0.00      0.00      0.00       185

    accuracy                           0.65    649947
   macro avg       0.57      0.41      0.44    649947
weighted avg       0.91      0.65      0.69    649947



d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Downloads\Mestrado\Experimentos\env\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Dow

# Média absolute threshold

In [6]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 21: 0.7265457388953662


# CD detectados

In [7]:
display(pd.DataFrame(cd_test_index))

,0,1,2,3,4,5,6,7,8,9,...,206686,206687,206688,206689,206690,206691,206692,206693,206694,206695
0,3,10,11,19,31,33,37,39,40,44,...,710024.0,710031.0,710034.0,710036.0,710038.0,710040.0,710043.0,710045.0,710048.0,710049.0
1,13,18,19,24,33,34,42,51,57,62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,15,16,19,30,33,55,57,80,83,91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,5,6,12,19,21,27,33,46,54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,33,46,323,430,516,1020,1106,1147,1477,1481,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4,19,33,57,77,105,107,157,193,200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
train = pd.read_csv('../../../train_CADE_cod_joined_attacks.csv')
test = pd.read_csv('../../../test_CADE_cod_joined_attacks.csv')

In [18]:
for i in range(len(filenames)):
    cd_test = test.loc[cd_test_index[i]]
    n_samples = test[test['Label'] == labels_str[filenames[i]]].shape[0]
    print(cd_test['Label'].value_counts())
    cd_test['Label'] = 'New Concept'
    retrain = pd.concat([train,cd_test], ignore_index=True)
    new_test = test.drop(cd_test_index[i])
    new_samples = train[train['Label'] == labels_str[filenames[i]]].sample(frac=n_samples/train[train['Label'] == labels_str[filenames[i]]].shape[0], random_state=123)
    new_test = pd.concat([new_test,new_samples],ignore_index=True)
    print(retrain['Label'].value_counts())
    print(new_test['Label'].value_counts())
    retrain.to_csv(f'retrain_hidden_{filenames[i]}_CADE_cod_joined_attacks.csv',index=False)
    new_test.to_csv(f'retest_hidden_{filenames[i]}_CADE_cod_joined_attacks.csv',index=False)

Label
DDoS             185839
Benign            14778
Infilteration      5366
DoS                 460
Bot                 193
BruteForce           47
Web                  13
Name: count, dtype: int64
Label
DDoS             808918
Benign           514148
DoS              418752
BruteForce       243804
New Concept      206696
Bot              183163
Infilteration    102810
Web                 595
Name: count, dtype: int64
Label
DDoS             319733
Benign           145892
DoS              130400
BruteForce        76141
Bot               57045
Infilteration     26761
Web                 172
Name: count, dtype: int64
Label
DoS              100917
Benign            25230
Infilteration      7484
DDoS                553
Bot                 114
Web                  14
BruteForce           14
Name: count, dtype: int64
Label
DDoS             808918
Benign           514148
DoS              418752
BruteForce       243804
Bot              183163
New Concept      134326
Infilteration    102810
We